# Frozen 20-Run PPO CUDA Training

This notebook resumes the frozen validation-selected PPO inventory. It does not evaluate the held-out test cohort.

> Research-only simulation. Not a medical device and not for clinical dosing or patient care.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import importlib
import json
import os
import shlex
import subprocess
import sys

def run_command(command, *, stage, cwd=None):
    command = [str(part) for part in command]
    print(f'\n[{stage}] $ {shlex.join(command)}')
    completed = subprocess.run(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    output = completed.stdout or ''
    if output:
        print(output, end='' if output.endswith('\n') else '\n')
    print(f'[{stage}] return code: {completed.returncode}')
    if completed.returncode != 0:
        tail = '\n'.join(output.splitlines()[-200:])
        raise RuntimeError(f'{stage} failed with return code {completed.returncode}. Last 200 lines:\n{tail}')
    return completed

repo = Path('/content/VitalDB-Feature-Selection')
remote = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
if not repo.exists():
    run_command(['git', 'clone', remote, str(repo)], stage='repository clone')
run_command(['git', '-C', str(repo), 'fetch', 'origin'], stage='repository fetch')
run_command(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], stage='repository sync')
required = '767f3bff3dcaeabc51049fba5ccba1ac02b69ae3'
run_command(['git', '-C', str(repo), 'merge-base', '--is-ancestor', required, 'HEAD'], stage='repository ancestry')
os.chdir(repo)
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()
head = run_command(['git', '-C', str(repo), 'rev-parse', 'HEAD'], stage='repository HEAD')
print('Repository commit:', head.stdout.strip())

In [ ]:
import pandas as pd
import torch

torch_before = torch.__version__
pandas_before = pd.__version__
dry_run = run_command(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', '-r', 'requirements-rl.txt'],
    stage='RL dependency dry run',
)
proposal = dry_run.stdout
if 'Would install torch-' in proposal or 'Would install pandas-' in proposal:
    raise RuntimeError('RL dependency profile would replace torch or pandas; aborting.')
run_command([
    sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3==2.9.0'
], stage='RL dependency install')
import stable_baselines3
import pandas as pd_after
import torch as torch_after
if torch_after.__version__ != torch_before or pd_after.__version__ != pandas_before:
    raise RuntimeError('torch/pandas changed during RL dependency setup.')
if not torch_after.cuda.is_available():
    raise RuntimeError('Full PPO notebook requires a CUDA runtime.')
print('SB3:', stable_baselines3.__version__)
print('Torch/CUDA preserved:', torch_after.__version__, torch_after.cuda.get_device_name(0))
print('Pandas preserved:', pd_after.__version__)

In [ ]:
drive_project = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
dataset_dir = drive_project / 'data/modeling/full'
project_data_root = drive_project / 'data'
protocol_dir = drive_project / 'outputs/ppo_protocol'
output_root = drive_project / 'outputs/ppo_control_comparison'
print('Drive modeling dataset:', dataset_dir)
print('Drive project data root:', project_data_root)
print('Dataset entries:', sorted(path.name for path in dataset_dir.iterdir()))
run_command([
    sys.executable, 'scripts/run_ppo_experiment.py',
    '--dataset-dir', str(dataset_dir),
    '--project-data-root', str(project_data_root),
    '--protocol-dir', str(protocol_dir),
    '--output-root', str(output_root),
    '--initialize-only',
], stage='PPO initialize-only preflight')
protocol = json.loads((protocol_dir / 'frozen_ppo_protocol.json').read_text())
print(json.dumps({
    'protocol_hash': protocol['protocol_hash'],
    'inventory': protocol['inventory'],
    'cohort': protocol['cohort'],
    'action': protocol['action'],
    'reward': protocol['reward'],
    'ppo': protocol['ppo'],
    'confirmation_text': protocol['confirmation_text'],
    'test_cohort_accessed': False,
}, indent=2))

In [ ]:
expected = protocol['confirmation_text']
confirmation = input(f'Type {expected} to start or resume the frozen CUDA inventory: ').strip()
if confirmation != expected:
    raise RuntimeError('Confirmation mismatch; full training remains locked.')
run_command([
    sys.executable, 'scripts/run_ppo_experiment.py',
    '--dataset-dir', str(dataset_dir),
    '--project-data-root', str(project_data_root),
    '--protocol-dir', str(protocol_dir),
    '--output-root', str(output_root),
    '--device', 'cuda',
    '--confirmation', confirmation,
], stage='frozen PPO CUDA inventory')

Completed runs are skipped and partial runs resume from `last_model.zip`. Validation selects checkpoints; the test cohort remains sealed.